# Lab 7.4 &mdash; Locating the Failure

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Walk the attribution ladder and name the step that actually caused a failure
- Handle the run with two things wrong &mdash; the ladder must name the first
- Count the wasted spend downstream of each failure
- Aggregate across runs and find the one fix worth making first

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Without a trace, every one of these is &lsquo;the agent hallucinated&rsquo;.**
> Five of the six are not, and each has a different owner.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- six failed runs, with enough trace to diagnose
# Each record is what a span tree would tell you. Without it, all six are reported the same way:
# "the agent got it wrong".

FAILED_RUNS = [
    {"id": "F1", "evidence_ok": False, "ignored_tool_error": False,
     "routed_to": "policy", "should_route_to": "policy", "constraints_dropped": False,
     "tokens_before_failure": 500, "tokens_total": 2400},
    {"id": "F2", "evidence_ok": True,  "ignored_tool_error": True,
     "routed_to": "ledger", "should_route_to": "ledger", "constraints_dropped": False,
     "tokens_before_failure": 380, "tokens_total": 1900},
    {"id": "F3", "evidence_ok": True,  "ignored_tool_error": False,
     "routed_to": "writer", "should_route_to": "sanctions", "constraints_dropped": False,
     "tokens_before_failure": 120, "tokens_total": 1730},
    {"id": "F4", "evidence_ok": True,  "ignored_tool_error": False,
     "routed_to": "policy", "should_route_to": "policy", "constraints_dropped": True,
     "tokens_before_failure": 800, "tokens_total": 2100},
    {"id": "F5", "evidence_ok": True,  "ignored_tool_error": False,
     "routed_to": "policy", "should_route_to": "policy", "constraints_dropped": False,
     "tokens_before_failure": 2000, "tokens_total": 2050},
    # two things wrong at once -- the ladder must name the one that came first
    {"id": "F6", "evidence_ok": False, "ignored_tool_error": False,
     "routed_to": "writer", "should_route_to": "policy", "constraints_dropped": True,
     "tokens_before_failure": 400, "tokens_total": 2600},
]

print(f"{len(FAILED_RUNS)} failed runs to diagnose")

## Concept

&ldquo;The agent was wrong&rdquo; is not a diagnosis. It is what you are left with when you did not keep
the trace, and it lands on whoever owns the agent regardless of who owns the bug.

The ladder is ordered on purpose. A run can have several things wrong with it; the one that came
**first** is the cause, and everything after it was doomed anyway.

## Section 1 &mdash; The ladder

Five rungs, checked in order, from the deck.

In [ ]:
def attribute(run: dict) -> str:
    """Name the step that caused this failure. Order matters: the first broken thing wins."""
    if not run["evidence_ok"]:
        return "retrieval"
    if run["ignored_tool_error"]:
        return "tool contract"
    if run["routed_to"] != run["should_route_to"]:
        return "routing"
    if run["constraints_dropped"]:
        return "handoff"
    return "generation"


def owner(step: str) -> str:
    """Who picks this up, which is the reason the diagnosis matters at all."""
    return {"retrieval": "the corpus and the chunker",
            "tool contract": "whoever wrote the tool",
            "routing": "the supervisor's descriptions",
            "handoff": "the message between two agents",
            "generation": "the prompt, or the model"}[step]

In [ ]:
# --- Self-check: Section 1
def find(rid):
    return next(r for r in FAILED_RUNS if r["id"] == rid)

check("bad evidence is a retrieval failure",
      lambda: attribute(find("F1")) == "retrieval")
check("an ignored tool error is a tool contract failure",
      lambda: attribute(find("F2")) == "tool contract")
check("the wrong specialist is a routing failure",
      lambda: attribute(find("F3")) == "routing")
check("a dropped constraint is a handoff failure",
      lambda: attribute(find("F4")) == "handoff")
check("only when everything upstream was fine is it generation",
      lambda: attribute(find("F5")) == "generation",
      "one run out of six -- and it is the diagnosis all six would have received")
check("F6 HAS THREE THINGS WRONG AND IS ATTRIBUTED TO THE FIRST",
      lambda: attribute(find("F6")) == "retrieval",
      "fixing its routing would change nothing: the evidence was already wrong when it routed")
check("every diagnosis names an owner",
      lambda: all(owner(attribute(r)) for r in FAILED_RUNS))

def _diagnose():
    for r in FAILED_RUNS:
        step = attribute(r)
        print(f"  {r['id']}  {step:15} -> {owner(step)}")
guard(_diagnose)

## Section 2 &mdash; What the failure cost

Everything spent after the failing step answered the wrong question. That number is what turns a
diagnosis into a priority.

In [ ]:
def wasted(run: dict) -> int:
    """Tokens spent after the thing that had already gone wrong."""
    return run["tokens_total"] - run["tokens_before_failure"]


def waste_rate(run: dict) -> float:
    return wasted(run) / run["tokens_total"] if run["tokens_total"] else 0.0

In [ ]:
# --- Self-check: Section 2
check("an early failure wastes most of the run",
      lambda: waste_rate(find("F3")) > 0.9,
      "a misroute at 120 tokens leaves 1,610 spent on the wrong specialist")
check("a late failure wastes almost nothing",
      lambda: waste_rate(find("F5")) < 0.05,
      "the generation failure happened at the end, so nothing downstream was thrown away")
check("waste is never negative",
      lambda: all(wasted(r) >= 0 for r in FAILED_RUNS))
check("the earliest failures are the most expensive ones",
      lambda: waste_rate(find("F3")) > waste_rate(find("F4")) > waste_rate(find("F5")),
      "which is why the ladder is ordered upstream-first, and why routing is worth measuring")
check("the total waste across the six runs is substantial",
      lambda: sum(wasted(r) for r in FAILED_RUNS)
              > 0.5 * sum(r["tokens_total"] for r in FAILED_RUNS))

## Section 3 &mdash; Which fix first

Six runs is not a sample, but the machinery is the point: group the failures, add up what each
group costs, and let that choose the work.

In [ ]:
def by_step() -> dict:
    """{step: {"runs": n, "wasted": tokens}} across every failed run."""
    out = {}
    for run in FAILED_RUNS:
        step = attribute(run)
        entry = out.setdefault(step, {"runs": 0, "wasted": 0})
        entry["runs"] += 1
        entry["wasted"] += wasted(run)
    return out


def worst_step() -> str:
    """The step to fix first: the one that wastes the most, not the one that fails most often."""
    return max(by_step().items(), key=lambda kv: kv[1]["wasted"])[0]

In [ ]:
# --- Self-check: Section 3
check("every failed run is accounted for exactly once",
      lambda: sum(v["runs"] for v in by_step().values()) == len(FAILED_RUNS))
check("the wasted tokens add up",
      lambda: sum(v["wasted"] for v in by_step().values())
              == sum(wasted(r) for r in FAILED_RUNS))
check("retrieval is the biggest single cause here",
      lambda: worst_step() == "retrieval")
check("and it is not the most frequent step, it is the most expensive one",
      lambda: by_step()["retrieval"]["runs"] == 2)
check("generation is the rarest cause, and the cheapest",
      lambda: by_step()["generation"]["runs"] == 1
              and by_step()["generation"]["wasted"] == min(v["wasted"]
                                                           for v in by_step().values()))
check("without the ladder every one of these is 'generation'",
      lambda: len(by_step()) > 1,
      "five different owners, one default diagnosis, and four teams who never hear about it")

def _priority():
    print(f"  {'step':16}{'runs':>6}{'wasted':>9}")
    print("  " + "-" * 32)
    for step, v in sorted(by_step().items(), key=lambda kv: -kv[1]["wasted"]):
        print(f"  {step:16}{v['runs']:>6}{v['wasted']:>9}")
    print(f"\n  fix first: {worst_step()} -- {owner(worst_step())}")
guard(_priority)

## Run it for real

Give the model one failed run in raw form and ask it to diagnose. The question is whether it
reaches for the ladder or for the default.

In [ ]:
if llm_ready():
    def _ask_diagnosis():
        run = find("F6")
        reply = ask(
            "An agent run produced a wrong answer. Here is what the trace shows:\n"
            f"- the retrieved evidence did not contain the answer: {not run['evidence_ok']}\n"
            f"- a tool returned an error the agent ignored: {run['ignored_tool_error']}\n"
            f"- routed to {run['routed_to']}, should have been {run['should_route_to']}\n"
            f"- the handoff dropped a constraint: {run['constraints_dropped']}\n\n"
            "Name the ONE step that should be fixed first, and why.",
            system="Be brief and name a single step.")
        print("  model:", reply.strip()[:220])
        print(f"  ladder: {attribute(run)} -- {owner(attribute(run))}")
    guard(_ask_diagnosis)

### Read it

F6 has three things wrong with it, and only one of them is worth fixing first. If the model picks
routing or the handoff, it has picked a real bug whose repair would have changed nothing about
this run &mdash; the evidence was already wrong before either of them happened.

That is the value of an ordered ladder over a judgement: it is not smarter, it is just consistent,
and consistency is what lets you aggregate across a thousand runs and act on the total.

In [ ]:
score()

## Your turn

1. The ladder assumes each rung is observable. Which of the five would your current system
   actually be able to answer from its logs today? That list is your instrumentation backlog.
2. `wasted` counts tokens. Count seconds instead, using Lab 7.2's `self_time`, and see whether the
   priority order changes. It usually does.
3. Add a sixth rung for a failure this ladder cannot express &mdash; a case where the run was correct
   and the *question* was wrong. Where does it go, and who owns it?